# Jetson AGX Orin benchmark - 11/02/2026

CLIP ViT-L/14-336 encoder + MLP projector, swept over **camera views** instead of
batch size. One *iteration* is one nuScenes keyframe taken from each of the 6
cameras; each camera is a *sub-iteration* run on its own, batch size 1, with no
`cuda.synchronize()` between them. Two runs of the same sweep:

| run | instrument | what it measures |
|---|---|---|
| `cpu_pure_latency` | `perf_counter_ns` | host time - launch cost for encode/project |
| `gpu_pure_latency` | CUDA events | GPU execution time, plus the iteration's GPU span |
| `nsys_capture` | NVTX ranges | host-side ranges, cross-check on the above |

The `iter` rows carry `view = -1`: they span the whole rig rather than one camera.
For the GPU run, `iter` is the span from the first view's first kernel to the last
view's last kernel, so **`iter` minus the summed sub-iterations is GPU idle** -
time the stream sat starved while the host decoded the next camera.

### Import Dependencies

In [ ]:
import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Set Path and Define Stages

In [ ]:
RESULTS = Path("../results/benchmark_11-2-26")

HOST = RESULTS / "cpu_pure_latency"      # perf_counter_ns
GPU = RESULTS / "gpu_pure_latency"       # CUDA events

STAGES = ["decode", "preprocess", "encode", "project"]
SPANS = ["subiter", "iter", "subtest"]

## run.warmup_iters ran outside the timed region, so iteration 0 is already warm.
## Kept so a run without that warmup can be trimmed the same way.
WARMUP = 0

## nuScenes keyframes are 2 Hz, so an iteration has 500 ms to process the rig
## before the harness falls behind the sensor
BUDGET_MS = 500.0

### Plot Style

Categorical hues are assigned in fixed slot order, never cycled. Stacked segments
carry direct value labels: at four series labels are mandatory, and two of these
slots sit below 3:1 contrast on this surface.

In [ ]:
## Categorical slots 1-4, fixed order
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
STAGE_COLOR = dict(zip(STAGES, SERIES))

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"

## White on a fill, except on yellow where it would not hold contrast
FILL_LABEL = {"#eda100": INK}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "font.size": 10, "figure.dpi": 110,
    "lines.linewidth": 2.0,
})

### Run Configuration

In [ ]:
manifest = json.loads((HOST / "manifest.json").read_text())

VIEWS = max(manifest["views"])
VIEW_ORDER = manifest["view_order"][:VIEWS]
IMAGES = sum(manifest["images_by_views"].values())
FRAMES = IMAGES // VIEWS

print(f"profile     : {manifest['profile']}  ({manifest['gpu']}, nvpmodel {manifest['nvpmodel_mode']})")
print(f"precision   : {manifest['precision']}   layer_index {manifest['layer_index']}  keep_cls {manifest['keep_cls']}")
print(f"views       : {manifest['views']}")
print(f"view order  : {VIEW_ORDER}")
print(f"workload    : {FRAMES} frames x {VIEWS} views = {IMAGES} images")
print(f"versions    : torch {manifest['torch']}, transformers {manifest['transformers']}")
print(f"git sha     : {manifest['git_sha']}   nsys_profiled {manifest['nsys_profiled']}")

### Combine CSV Stage Data

Every stage CSV shares the schema `(views, iter, view, duration_ms)`. `view` is the
camera index inside an iteration, or `-1` for a row that spans the whole rig.

In [ ]:
def load_run(run_dir, stages):
    frames = []
    for stage in stages:
        csv = Path(run_dir) / f"{stage}.csv"
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        df["stage"] = stage
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


## Rows for one stage, any run
def stage(long, name):
    return long[long["stage"] == name]


## One row per (iteration, view) with a column per stage. Drops the view == -1
## spans, which belong to the iteration rather than to any camera.
def to_wide(long, run_name):
    present = [s for s in STAGES if s in set(long["stage"])]
    return (
        long[long["view"] >= 0]
        .pivot_table(index=["views", "iter", "view"], columns="stage",
                     values="duration_ms")
        .reindex(columns=present)
        .rename(columns=lambda s: f"{s}_ms")
        .reset_index()
        .assign(run=run_name)
    )

In [ ]:
host_long = load_run(HOST, STAGES + SPANS)
gpu_long = load_run(GPU, STAGES + SPANS)

host_wide = to_wide(host_long, "host")
gpu_wide = to_wide(gpu_long, "gpu")

print("host stages:", sorted(set(host_long["stage"])))
print("gpu stages :", sorted(set(gpu_long["stage"])))
host_wide.head(3)

### Read NSYS SQLite Data

Read from the capture's SQLite rather than the `nsys stats` CSV: that report
prefixes every range name with its range-stack marker, which the harness' own
parser does not expect.

In [ ]:
QUERY = """
SELECT text AS name, start, end
FROM NVTX_EVENTS
WHERE eventType = 59          -- NvtxPushPopRange
  AND text LIKE '%/v=%'
ORDER BY start
"""

with sqlite3.connect(RESULTS / "nsys_capture.sqlite") as conn:
    nvtx = pd.read_sql_query(QUERY, conn)

nvtx["duration_ms"] = (nvtx["end"] - nvtx["start"]) / 1e6

## Same name format as push() in benchmark_tests/deep_metrics.py:
## "<stage>/v=<views>/view=<view>"
parts = nvtx["name"].str.extract(
    r"^(?P<stage>\w+)/v=(?P<n_views>\d+)/view=(?P<view>-?\d+)$")
nvtx = nvtx.join(parts).dropna(subset=["stage"])
nvtx = nvtx.astype({"n_views": int, "view": int})

## The range name carries no iteration index, so it is the position within its
## (stage, view) group -- valid because the query returns them in start order
nvtx["iter"] = nvtx.groupby(["n_views", "stage", "view"]).cumcount()

print(f"{len(nvtx)} ranges, stages: {sorted(set(nvtx['stage']))}")
nvtx[["stage", "view", "iter", "duration_ms"]].head(5)

## Timings per View (sub-iteration)

Each bar is one camera's mean sub-iteration, split by stage. The host panel shows
all four stages; the GPU panel has only encode and project, because decode and
preprocess launch no kernels and an event pair around them would read ~0.

In [ ]:
host_warm = host_wide[host_wide["iter"] >= WARMUP]
gpu_warm = gpu_wide[gpu_wide["iter"] >= WARMUP]

host_by_view = host_warm.groupby("view")[[f"{s}_ms" for s in STAGES]].mean()
gpu_stages = [s for s in STAGES if f"{s}_ms" in gpu_warm.columns]
gpu_by_view = gpu_warm.groupby("view")[[f"{s}_ms" for s in gpu_stages]].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True,
                               layout="constrained")
y = np.arange(VIEWS)
ticks = [f"{i}  {n}" for i, n in enumerate(VIEW_ORDER)]

for ax, table, stages, title in (
    (ax1, host_by_view, STAGES, "host time per view (perf_counter)"),
    (ax2, gpu_by_view, gpu_stages, "GPU time per view (CUDA events)"),
):
    left = np.zeros(len(table))
    for s in stages:
        vals = table[f"{s}_ms"].to_numpy()
        ax.barh(y, vals, left=left, height=0.62, color=STAGE_COLOR[s],
                edgecolor=SURFACE, linewidth=2, label=s)
        for yi, (v, l) in enumerate(zip(vals, left)):
            if v >= 3.0:        # only where the segment can hold a label
                ax.text(l + v / 2, yi, f"{v:.1f}", ha="center", va="center",
                        fontsize=9, color=FILL_LABEL.get(STAGE_COLOR[s], "#ffffff"))
        left += vals

    ## total at the bar end, in ink rather than a series color
    for yi, total in enumerate(left):
        ax.text(total + left.max() * 0.015, yi, f"{total:.1f} ms",
                va="center", fontsize=9, color=INK_2)

    ax.set_xlim(0, left.max() * 1.16)
    ax.set_title(title)
    ax.set_xlabel("mean ms per sub-iteration")
    ax.grid(axis="y", visible=False)

ax1.set_yticks(y, ticks)
ax1.invert_yaxis()

## One legend for both panels, outside the axes so it cannot sit on a bar
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=len(STAGES))
plt.show()

Spread per camera, same measure, so one hue. A camera that decoded a larger JPEG
or hit a thermal stall would show up here as a wider box or outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for ax, long, title in (
    (axes[0], host_long, "host sub-iteration per view"),
    (axes[1], gpu_long, "GPU sub-iteration per view"),
):
    sub = stage(long, "subiter")
    data = [sub.loc[sub["view"] == v, "duration_ms"].to_numpy() for v in range(VIEWS)]
    bp = ax.boxplot(data, patch_artist=True, widths=0.55,
                    medianprops=dict(color=SURFACE, linewidth=2),
                    flierprops=dict(marker="o", markersize=4,
                                    markerfacecolor=MUTED, markeredgecolor="none"))
    for patch in bp["boxes"]:
        patch.set(facecolor=SERIES[0], edgecolor=SURFACE, linewidth=2)
    for part in ("whiskers", "caps"):
        for line in bp[part]:
            line.set(color=AXIS, linewidth=1.2)

    ax.set_xticks(np.arange(1, VIEWS + 1), [str(v) for v in range(VIEWS)])
    ax.set_xlabel("view index")
    ax.set_ylabel("ms")
    ax.set_title(title)
    ax.grid(axis="x", visible=False)

fig.tight_layout()
plt.show()

## Timings per Iteration

One iteration is all 6 cameras. Three series on one scale:

- **host iteration** - wall time on the host for the whole rig
- **GPU span** - first kernel of view 0 to last kernel of view 5
- **GPU busy** - the six sub-iterations summed, i.e. kernels actually executing

The dashed line is the 2 Hz keyframe budget. GPU span above GPU busy is idle time.

In [ ]:
host_iter = stage(host_long, "iter").set_index("iter")["duration_ms"].sort_index()
gpu_iter = stage(gpu_long, "iter").set_index("iter")["duration_ms"].sort_index()
gpu_busy = stage(gpu_long, "subiter").groupby("iter")["duration_ms"].sum().sort_index()
host_busy = stage(host_long, "subiter").groupby("iter")["duration_ms"].sum().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.4), layout="constrained",
                               gridspec_kw={"width_ratios": [3.2, 1]})

## host iteration and GPU span land within ~1 ms of each other, so the host line
## goes underneath, thicker and softened, and the span rides on top of it
ax1.plot(host_iter.index, host_iter.to_numpy(), color=SERIES[0],
         linewidth=3.2, alpha=0.45, label="host iteration")
ax1.plot(gpu_iter.index, gpu_iter.to_numpy(), color=SERIES[1],
         linewidth=1.8, label="GPU span")
ax1.plot(gpu_busy.index, gpu_busy.to_numpy(), color=SERIES[2],
         linewidth=1.8, label="GPU busy")

lo = min(gpu_busy.min(), gpu_iter.min(), host_iter.min())
hi = max(gpu_iter.max(), host_iter.max())
pad = (hi - lo) * 0.14
ax1.set_ylim(lo - pad, hi + pad)
ax1.set_xlim(host_iter.index.min(), host_iter.index.max())
ax1.set_xlabel("iteration (one nuScenes keyframe, all 6 views)")
ax1.set_ylabel("ms")
ax1.set_title(f"per-iteration timing, {VIEWS} views")

## A single ratio against a limit is a meter, not a line on a budget-scaled axis
mean_iter = host_iter.mean()
ax2.barh([0], [BUDGET_MS], height=0.46, color=GRID)
ax2.barh([0], [mean_iter], height=0.46, color=SERIES[0])
ax2.text(mean_iter / 2, 0, f"{mean_iter:.0f} ms", ha="center", va="center",
         fontsize=10, color="#ffffff")
ax2.text(BUDGET_MS, 0.32, f"2 Hz budget {BUDGET_MS:.0f} ms",
         ha="right", va="bottom", fontsize=9, color=INK_2)
ax2.set_xlim(0, BUDGET_MS * 1.03)
ax2.set_ylim(-1.7, 1.3)     # keeps the track a band, not a block
ax2.set_yticks([])
ax2.set_xlabel("ms")
ax2.set_title("mean iteration vs real-time budget")
ax2.grid(visible=False)
ax2.spines["left"].set_visible(False)

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=3)
plt.show()

The same iteration split into the part where kernels run and the part where the
GPU waits on the host. The two stack to the GPU span above.

In [ ]:
common = gpu_iter.index.intersection(gpu_busy.index)
busy = gpu_busy.loc[common].to_numpy()
idle = (gpu_iter.loc[common] - gpu_busy.loc[common]).to_numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.2),
                               gridspec_kw={"width_ratios": [3, 1]})

ax1.stackplot(common.to_numpy(), busy, idle,
              colors=[SERIES[0], SERIES[1]],
              labels=["GPU busy", "GPU idle (waiting on host)"],
              edgecolor=SURFACE, linewidth=1.5)
ax1.set_xlabel("iteration")
ax1.set_ylabel("ms")
ax1.set_title("GPU span per iteration, busy vs idle")
ax1.legend(loc="lower right", ncol=2)
ax1.set_xlim(common.min(), common.max())

share = 100 * idle / (busy + idle)
ax2.hist(share, bins=24, color=SERIES[1], edgecolor=SURFACE, linewidth=1.5)
ax2.axvline(share.mean(), color=INK_2, linestyle="--", linewidth=1.5)
ax2.text(share.mean(), ax2.get_ylim()[1], f" mean {share.mean():.1f}%",
         va="top", fontsize=9, color=INK_2)
ax2.set_xlabel("GPU idle, % of iteration")
ax2.set_ylabel("iterations")
ax2.set_title("idle share")
ax2.grid(axis="x", visible=False)

fig.tight_layout()
plt.show()

### Compute Metrics from Data

In [ ]:
def describe(long, run, stages):
    rows = []
    for s in stages:
        d = stage(long, s)
        if d.empty:
            continue
        rows.append({
            "run": run, "stage": s, "n": len(d),
            "mean_ms": d["duration_ms"].mean(),
            "median_ms": d["duration_ms"].median(),
            "p95_ms": d["duration_ms"].quantile(0.95),
            "max_ms": d["duration_ms"].max(),
        })
    return pd.DataFrame(rows)


per_view = pd.concat([
    describe(host_long, "host (perf_counter)", STAGES + ["subiter"]),
    describe(gpu_long, "gpu (cuda events)", STAGES + ["subiter"]),
    describe(nvtx, "nvtx (host ranges)", STAGES + ["subiter"]),
], ignore_index=True)

per_view.round(3)

In [ ]:
## Side by side, so the host/GPU/NVTX reading of each stage lines up
per_view.pivot_table(index="stage", columns="run", values="mean_ms") \
        .reindex(STAGES + ["subiter"]).round(3)

In [ ]:
subtest_s = stage(host_long, "subtest")["duration_ms"].iloc[0] / 1000
idle_ms = (gpu_iter - gpu_busy).mean()

## Mean of the per-iteration shares, the same quantity the histogram above marks
idle_share = (100 * (gpu_iter - gpu_busy) / gpu_iter).mean()

print("per iteration (all %d views)" % VIEWS)
print(f"  host iteration : {host_iter.mean():8.2f} ms   (p95 {host_iter.quantile(0.95):.2f})")
print(f"  GPU span       : {gpu_iter.mean():8.2f} ms   (p95 {gpu_iter.quantile(0.95):.2f})")
print(f"  GPU busy       : {gpu_busy.mean():8.2f} ms")
print(f"  GPU idle       : {idle_ms:8.2f} ms   ({idle_share:.1f}% of the span)")
print()
print("throughput")
print(f"  sub-test wall  : {subtest_s:8.2f} s for {IMAGES} images")
print(f"  images/s       : {IMAGES / subtest_s:8.2f}")
print(f"  frames/s       : {FRAMES / subtest_s:8.2f}   (one frame = {VIEWS} views)")
print(f"  2 Hz budget    : {BUDGET_MS:8.2f} ms per frame")
print(f"  headroom       : {100 * (BUDGET_MS - host_iter.mean()) / BUDGET_MS:8.1f}% under budget")

### Findings

Read off the cells above for this run:

1. **The iteration is host-bound.** The GPU span tracks the host iteration almost
   exactly while GPU busy sits far below it, so the stream is starved rather than
   saturated - over a third of each iteration is idle GPU.
2. **No camera is an outlier, but the views alternate.** Odd view indices run a
   little slower than even ones, and the gap is entirely in `preprocess` - decode
   and encode are flat across all six. It is a small effect, but it is systematic
   rather than noise, and it is consistent with the pageable-copy stall in point 3
   landing on alternate views rather than with anything about the cameras.
3. **`preprocess` is the host cost that matters**, and it is larger than it looks:
   `data.preprocess` ends in a `.to(device)` on a pageable tensor, and CUDA
   serializes the stream before a pageable host-to-device copy. So that number
   absorbs the wait for the previous view's encode. Pinned memory plus
   `non_blocking=True` is the change that would recover the idle time - at which
   point preprocess becomes asynchronous too and needs its own event pair, like
   encode and project.
4. **Real-time holds at 6 views** against the 2 Hz keyframe rate, with the headroom
   printed above - but that margin is what pinning would widen.